<a href="https://colab.research.google.com/github/cheecaixi/AISDPJ/blob/main/AISDPJ_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Load Data
2. Remove Duplicates
3. Standardise Categorical Labels
4. Fix Invalid Values
5. Handle Missing Values
6. Correct Data Types
7. Detect and Handle Outliers
8. Encode Categorical Features
9. Scale Numerical Features
10. Feature Engineering
11. Final Validation

# **1. Load Data**

In [ ]:
# Import Libraries
import sqlite3
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
# Connect to your local database file
DB_PATH = os.path.join("data", "gas_monitoring.db")
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM gas_monitoring", conn)
conn.close()

In [ ]:
display(df)

In [ ]:
# The inital Shape of the dataset before cleaning
print(df.shape)

In [ ]:
print("\nDataset Summary:")

# Re-identify numerical and categorical columns after cleaning
numerical_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(include='object').columns

summary = pd.DataFrame({
    'Dtype': df.dtypes,
    'Non-Null Count': df.count(),
    'Missing Values': df.isnull().sum()
})

summary['Column Type'] = 'Other'
summary.loc[numerical_cols, 'Column Type'] = 'Numerical'
summary.loc[categorical_cols, 'Column Type'] = 'Categorical'

# Drop the 'Dtype' column as requested
summary = summary.drop(columns=['Dtype'])

display(summary)

In [ ]:
# Visualise missing values by column

missing_counts = df.isnull().sum()

missing_counts = missing_counts[missing_counts > 0]

plt.figure(figsize=(8,4))

missing_counts.sort_values().plot(kind="barh")

plt.title("Missing Values by Column")
plt.xlabel("Missing Count")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

Displayed a summary table of the dataset, showing each column's non-null count, missing values, and column type.

The dataset contains 10,000 records with a mix of categorical and numerical features related to environmental and sensor data.

Most columns are complete with no missing values, indicating generally good data quality.

However, several attributes such as
- Humidity (1,928 missing values)
- MetalOxideSensor_Unit2 (1,410 missing values)
- Ambient Light Level (1,054 missing values)
- CO_GasSensor (834 missing values)
- Ambient_Light_Ordinal (1054 missing values)

require data cleaning or handling before analysis.

The dataset also includes engineered features like CO2_Disagreement, MOS_Mean, Is_Night, and Activity_Encoded, which may help improve predictive analysis and pattern detection.

# **2. Remove Duplicates**

In [ ]:
# Remove duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate Rows Found: {duplicate_count}")

df = df.drop_duplicates()
print(f"Shape After Removing Duplicates: {df.shape}")

Removing duplicates helps improve data accuracy and prevents repeated records from affecting the reliability of the analysis and model results.

In [ ]:
# Visualise duplicate removal
plt.figure(figsize=(5, 4))
duplicate_count = 265

plt.bar(
    ["Duplicates Removed", "Remaining Records"],
    [duplicate_count, len(df)]
)

plt.title("Duplicate Record Analysis")
plt.ylabel("Count")

plt.show()

# **3. Standardize Categories**

In [ ]:
categorical_columns = [
    "HVAC Operation Mode",
    "Activity Level"
]

print("Category Counts BEFORE Standardization\n")

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

In [ ]:
# HVAC Operation Mode
df["HVAC Operation Mode"] = (
    df["HVAC Operation Mode"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Activity Level
df["Activity Level"] = (
    df["Activity Level"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Merge concatenated variants to snake_case mapping
df["Activity Level"] = df["Activity Level"].replace({
    "lowactivity": "low_activity",
    "moderateactivity": "moderate_activity",
    "highactivity": "high_activity"
})

In [ ]:
# Display all unique values and counts for categorical columns

categorical_columns = [
    "Time of Day",
    "HVAC Operation Mode",
    "Ambient Light Level",
    "Activity Level"
]

for col in categorical_columns:
    print(f"{col}")
    print(f"Unique Categories: {df[col].nunique()}\n")
    print(df[col].value_counts(dropna=False))

In [ ]:
# Visualise category counts before standardisation

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_columns):

    df[col].value_counts().plot(
        kind="bar",
        ax=axes[i]
    )

    axes[i].set_title(col)
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()

Several category inconsistencies were identified in the HVAC Operation Mode and Activity Level features. Variations caused by capitalization, underscores, and inconsistent naming conventions were standardized to ensure that identical categories were represented consistently. This reduces redundancy and improves the quality of subsequent encoding and analysis.

# **4. Fix Invalid Values**

In [ ]:
CONTAMINATED_SESSIONS = [2586]
MIN_TEMPERATURE, MAX_TEMPERATURE = 18.0, 40.0
MIN_HUMIDITY, MAX_HUMIDITY = 0.0, 100.0

# Create a clear tracking point
df_working = df.copy()
print(f"Initial Row Count: {len(df_working):,}\n")

In [ ]:
rows_before = len(df_working)
# Remove contaminated session
df_working = df_working[~df_working["Session ID"].isin(CONTAMINATED_SESSIONS)]

# Temperature physical boundaries
df_working = df_working[(df_working["Temperature"] >= MIN_TEMPERATURE) & (df_working["Temperature"] <= MAX_TEMPERATURE)]

# Humidity boundaries (preserving pre-existing NaNs)
df_working = df_working[((df_working["Humidity"] >= MIN_HUMIDITY) & (df_working["Humidity"] <= MAX_HUMIDITY)) | (df_working["Humidity"].isna())]

# Negative gas sensor values check (preserving pre-existing NaNs)
df_working = df_working[(df_working["CO2_InfraredSensor"] >= 0) | (df_working["CO2_InfraredSensor"].isna())]
df_working = df_working[(df_working["CO_GasSensor"] >= 0) | (df_working["CO_GasSensor"].isna())]

print(f"Removed {rows_before - len(df_working):,} invalid rows.")
print(f"Current Row Count: {len(df_working):,}\n")

Remove physically impossible sensor readings that may reduce data quality and negatively affect model performance.

# **5. Handle Missing Values**

In [ ]:
print(f"\nMissing values: {df_working.isnull().sum().sum()}")

In [ ]:
# Overview of missing values
missing_counts = df_working.isnull().sum()
missing_pct = (missing_counts / len(df_working)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_pct.round(2)
}).query("`Missing Count` > 0")

print("Missing Values Summary:")
print(missing_summary)

Which columns has missing values, and the percentage of the missing values

In [ ]:
# Understand data distribution before choosing imputation methods
columns_with_missing = [
    "Humidity",
    "MetalOxideSensor_Unit2",
    "CO_GasSensor"
]

print("\nStatistics for Columns with Missing Values:")
print(
    df_working[columns_with_missing]
    .agg(["mean", "median", "std", "min", "max"])
    .round(4)
)

Understanding the distribution of numerical features and determine whether median or mean imputation is more suitable.

Visualize & Impute Humidity

In [ ]:
plt.figure(figsize=(8, 4))

# Plot baseline profile from raw backup
sns.histplot(data=df, x="Humidity", color="orange", alpha=0.4, kde=True, label="Pre-Imputation")

# Track count and execute your session median logic
missing_hum_before = df_working["Humidity"].isnull().sum()

df_working["Humidity"] = df_working.groupby("Session ID")["Humidity"].transform(lambda x: x.fillna(x.median()))
if df_working["Humidity"].isnull().sum() > 0:
    df_working["Humidity"] = df_working["Humidity"].fillna(df_working["Humidity"].median())

# Overlay the clean imputed profile
sns.histplot(data=df_working, x="Humidity", color="teal", alpha=0.4, kde=True, label="Post-Imputation (Session Median)")

plt.title(f"Humidity Distribution Check (Filled {missing_hum_before:,} values)")
plt.legend()
plt.show()

print(f"Successfully imputed {missing_hum_before:,} missing records for Humidity.")

Visualize & Impute MetalOxideSensor_Unit2

In [ ]:
plt.figure(figsize=(8, 4))

# Plot baseline profile from raw backup
sns.histplot(data=df, x="MetalOxideSensor_Unit2", color="orange", alpha=0.4, kde=True, label="Pre-Imputation")

# Track count and execute your session median logic
missing_mos_before = df_working["MetalOxideSensor_Unit2"].isnull().sum()

df_working["MetalOxideSensor_Unit2"] = df_working.groupby("Session ID")["MetalOxideSensor_Unit2"].transform(lambda x: x.fillna(x.median()))
if df_working["MetalOxideSensor_Unit2"].isnull().sum() > 0:
    df_working["MetalOxideSensor_Unit2"] = df_working["MetalOxideSensor_Unit2"].fillna(df_working["MetalOxideSensor_Unit2"].median())

# Overlay the clean imputed profile
sns.histplot(data=df_working, x="MetalOxideSensor_Unit2", color="purple", alpha=0.4, kde=True, label="Post-Imputation (Session Median)")

plt.title(f"MetalOxideSensor_Unit2 Distribution Check (Filled {missing_mos_before:,} values)")
plt.legend()
plt.show()

print(f"Successfully imputed {missing_mos_before:,} missing records for MetalOxideSensor_Unit2.")

In [ ]:
plt.figure(figsize=(8, 4))

# Fill missing spaces with a text string temporarily just for plotting visibility
sns.countplot(data=df_working.fillna({"CO_GasSensor": "Missing"}), x="CO_GasSensor", palette="muted")
plt.title("CO Gas Sensor Discrete Distributions (Checking for Imputation Balance)")
plt.show()

# Apply your global median logic
missing_co_before = df_working["CO_GasSensor"].isnull().sum()
df_working["CO_GasSensor"] = df_working["CO_GasSensor"].fillna(df_working["CO_GasSensor"].median())

print(f"Successfully imputed {missing_co_before:,} discrete CO entries with global median.")

In [ ]:
plt.figure(figsize=(8, 4))

sns.countplot(data=df_working.fillna({"Ambient Light Level": "Missing"}), y="Ambient Light Level", palette="viridis")
plt.title("Ambient Light Level Categorical Proportions")
plt.show()

# Apply your global mode logic
missing_light_before = df_working["Ambient Light Level"].isnull().sum()
global_light_mode = df_working["Ambient Light Level"].mode()[0]
df_working["Ambient Light Level"] = df_working["Ambient Light Level"].fillna(global_light_mode)

print(f"Successfully imputed {missing_light_before:,} missing rows with Global Mode: '{global_light_mode}'")

In [ ]:
plt.figure(figsize=(10, 3))
sns.heatmap(df_working[["Humidity", "MetalOxideSensor_Unit2", "CO_GasSensor", "Ambient Light Level"]].isnull(), 
            cbar=False, yticklabels=False, cmap="viridis")
plt.title("Null Value Heatmap Post-Pipeline (Solid color means 0 missing values remain!)")
plt.show()

print(f"Final Data completeness check. Remaining Null entries: {df_working.isnull().sum().sum()}")

# **6. Correct Data Types**

In [ ]:
# Record dtypes BEFORE correction for comparison
dtypes_before = df.dtypes.copy()
memory_before = df.memory_usage(deep=True).sum()

print("Data types BEFORE correction:")
print(dtypes_before)

In [ ]:
# Convert sensor and numerical columns to numeric format
float_columns = [
    "Temperature", "Humidity",
    "CO2_InfraredSensor",
    "CO2_ElectroChemicalSensor",
    "MetalOxideSensor_Unit1",
    "MetalOxideSensor_Unit2",
    "MetalOxideSensor_Unit3",
    "MetalOxideSensor_Unit4",
    "CO_GasSensor"
]

for col in float_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Session ID is treated as an identifier rather than a numerical feature
df["Session ID"] = df["Session ID"].astype(str)

# Convert categorical text features to category dtype
categorical_columns = [
    "Time of Day",
    "HVAC Operation Mode",
    "Ambient Light Level",
    "Activity Level"
]

for column in categorical_columns:
    df[column] = df[column].astype("category")

print("Data Types After Correction:")
print(df.dtypes)

In [ ]:
# Compare data types before and after correction
dtype_comparison = pd.DataFrame({
    "Before": dtypes_before,
    "After": df.dtypes
})

print(dtype_comparison)

In [ ]:
# Memory usage comparison: before vs after dtype corrections
memory_after = df.memory_usage(deep=True).sum()

print(f"Memory BEFORE dtype correction : {memory_before / 1024:.1f} KB")
print(f"Memory AFTER  dtype correction : {memory_after  / 1024:.1f} KB")
print(f"Memory saved                   : {(memory_before - memory_after) / 1024:.1f} KB "
      f"({100 * (memory_before - memory_after) / memory_before:.1f}% reduction)")

In [ ]:
# Final validation after dtype correction
coercion_nulls = df[float_columns].isnull().sum()

if coercion_nulls.sum() > 0:
    print("Warning - coercion introduced NaNs in:")
    print(coercion_nulls[coercion_nulls > 0])
else:
    print("No new NaNs introduced by dtype coercion.")

print(f"\nFinal dataset shape: {df.shape}")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

Several features were converted to more appropriate data types based on their role within the dataset. Numerical sensor measurements were stored as numeric values, categorical variables were converted to category dtype, and binary indicators were optimized using smaller integer types. These changes improved memory efficiency while preserving data integrity, and validation confirmed that no new missing values were introduced during the conversion process.

# **7. Detect and Handle Outliers**

In [ ]:
import matplotlib.pyplot as plt

# Visualise outliers with boxplots before capping

# Columns to check for statistical outliers (copied from a later cell for self-containment)
all_outlier_columns = [
    "Temperature", "Humidity", "CO2_InfraredSensor",
    "CO2_ElectroChemicalSensor", "MetalOxideSensor_Unit1",
    "MetalOxideSensor_Unit2", "MetalOxideSensor_Unit3",
    "MetalOxideSensor_Unit4", "CO_GasSensor",
    "CO2_Disagreement", "MOS_Mean"
]

# Filter to include only columns that exist in the DataFrame
outlier_columns = [col for col in all_outlier_columns if col in df.columns]

# Check if there are any columns left to plot
if not outlier_columns:
    print("No outlier columns found in DataFrame to plot.")
else:
    # Adjust subplot grid based on the number of actual outlier columns
    num_cols = len(outlier_columns)
    num_rows = (num_cols + 3) // 4 # Calculate rows needed for up to 4 columns per row
    fig, axes = plt.subplots(num_rows, 4, figsize=(16, num_rows * 3.5))
    axes = axes.flatten()

    for i, col in enumerate(outlier_columns):
        axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor="steelblue", alpha=0.6))
        axes[i].set_title(col, fontsize=9)
        axes[i].tick_params(axis="x", labelbottom=False)

    # Hide unused subplots
    for j in range(len(outlier_columns), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Boxplots Before Outlier Capping", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Columns to check for statistical outliers
# We skip encoded/flag columns (Is_Night, Activity_Encoded, Ambient_Light_Ordinal)
all_outlier_columns = [
    "Temperature", "Humidity", "CO2_InfraredSensor",
    "CO2_ElectroChemicalSensor", "MetalOxideSensor_Unit1",
    "MetalOxideSensor_Unit2", "MetalOxideSensor_Unit3",
    "MetalOxideSensor_Unit4", "CO_GasSensor",
    "CO2_Disagreement", "MOS_Mean"
]

# Filter to include only columns that exist in the DataFrame
outlier_columns = [col for col in all_outlier_columns if col in df.columns]

# IQR method: flag values below Q1 - 1.5*IQR or above Q3 + 1.5*IQR
outlier_report = {}
for col in outlier_columns:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {"lower": lower, "upper": upper, "outliers": n_out}

outlier_df = pd.DataFrame(outlier_report).T
print("Outlier summary (IQR method):")
print(outlier_df.to_string())

In [ ]:
# Visualise distributions before outlier handling
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_columns):

    axes[i].hist(
        df[col].dropna(),
        bins=30
    )

    axes[i].set_title(col, fontsize=9)

for j in range(len(outlier_columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions Before Outlier Handling")

plt.tight_layout()
plt.show()

In [ ]:
# Cap outliers using Winsorization (clip to IQR bounds)
# This keeps the row but pulls extreme values to the fence — no data is lost
for col in outlier_columns:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

print("Outlier capping complete.")
print(f"Shape after capping: {df.shape}")
print(f"Missing values after capping: {df.isnull().sum().sum()}")

In [ ]:
# Visualise distributions after outlier capping
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_columns):
    axes[i].hist(
        df[col].dropna(),
        bins=30
    )

    axes[i].set_title(col, fontsize=9)

for j in range(len(outlier_columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions After Outlier Capping")
plt.tight_layout()
plt.show()

In [ ]:
important_cols = [
    "Temperature",
    "Humidity",
    "CO2_InfraredSensor",
    "CO_GasSensor"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(important_cols):

    axes[i].hist(
        df[col],
        bins=30
    )

    axes[i].set_title(col)

plt.tight_layout()
plt.show()

Purpose: To identify and reduce the influence of extreme sensor readings that may negatively affect scaling and machine learning performance.

The histograms before outlier handling showed long tails and extreme observations in several sensor features, indicating the presence of statistical outliers. After applying IQR based capping, the extreme values were brought within acceptable ranges while preserving the overall shape of each distribution. This approach reduced the influence of outliers without removing any records from the dataset.

# **8. Encode Categorical Features**



In [ ]:
# Identify categorical columns
categorical_cols = [
    "Time of Day",
    "HVAC Operation Mode",
    "Activity Level"
]

print("Categorical Features:")
print(categorical_cols)

To identify categorical features that require numerical encoding before machine learning.

In [ ]:
# Apply one hot encoding
df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

print("Encoding complete.")
print(f"Updated dataset shape: {df.shape}")

To convert categorical labels into numerical format while avoiding ordinal assumptions between categories.

Categorical features were transformed into numerical representations using one hot encoding to ensure compatibility with machine learning algorithms. The first category was dropped during encoding to reduce redundancy and prevent multicollinearity between encoded variables.

# **9. Scale Numerical Features**

In [ ]:
# Columns to scale only continuous sensor readings
# Exclude: encoded flags (0/1), ordinal columns, Session ID
all_scale_cols = [
    "Temperature",
    "Humidity",
    "CO2_InfraredSensor",
    "CO2_ElectroChemicalSensor",
    "MetalOxideSensor_Unit1",
    "MetalOxideSensor_Unit2",
    "MetalOxideSensor_Unit3",
    "MetalOxideSensor_Unit4",
    "CO_GasSensor",
    "CO2_Disagreement",
    "MOS_Mean"
]

# Filter to include only columns that exist in the DataFrame
scale_cols = [col for col in all_scale_cols if col in df.columns]

# Fit and transform
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])

# Verify: scaled columns should have mean ≈ 0 and std ≈ 1
print("Post-scaling stats (should be mean≈0, std≈1):")
print(df_scaled[scale_cols].agg(["mean", "std"]).round(4).to_string())

In [ ]:
# Keep both: df (unscaled, interpretable) and df_scaled (ready for modelling)
# For tree-based models (Random Forest, XGBoost) scaling is not needed
# For distance/gradient-based models (KNN, SVM, LogReg) use df_scaled
print(f"df        — original scale, shape: {df.shape}")
print(f"df_scaled — standardised,    shape: {df_scaled.shape}")

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    df["CO2_InfraredSensor"],
    bins=30,
    alpha=0.6,
    label="Before Scaling"
)

plt.hist(
    df_scaled["CO2_InfraredSensor"],
    bins=30,
    alpha=0.6,
    label="After Scaling"
)

plt.title("CO2 Infrared Sensor Before and After Scaling")

plt.legend()

plt.show()

To standardize continuous sensor features so that variables with larger numerical ranges do not dominate machine learning models.

Continuous sensor features were standardized using StandardScaler while preserving a separate unscaled version of the dataset for interpretation. Scaling is particularly important for distance based and gradient based machine learning algorithms, while tree based models typically do not require feature scaling.